# Wireless Networking and Mobile Computing

### Prerequisites

This Jupyter Notebook has been tested with Visual Studio Code, running in a local Python environment.

**Installs**

In [ ]:
# --- Installs: ensure required packages are available ---
%pip install matplotlib numpy --quiet

**Imports**

In [ ]:
# --- Imports: libraries used in this notebook ---
from ipywidgets import FloatSlider, interact, HBox, VBox, interactive_output, Layout
import matplotlib.pyplot as plt
import numpy as np
import random 

### 1. Sources of Noise in Communication Networks

Noise plays a crucial role in communication networks because it affects the quality and reliability of data transmission. Noise is any unwanted signal that interferes with the transmission and reception of desired signals. Noise comes from various sources, such as thermal fluctuations, shot noise, or electromagnetic interference, and it affects the signal by increasing errors in the received data. There is a theoretical limit on how much information can be transmitted over a noisy channel. Mitigating noise through techniques such as error correction, filtering, and modulation are important in optimizing communication systems to achieve higher reliability and throughput.

#### 1.1 Noise Types
- Thermal Noise is caused by the random motion of electrons in a conductor, which generates a fluctuating voltage. It is present in all electrical devices and is proportional to temperature.

  $ N = kTB $, where:
  - $ N $ is the noise power,
  - $ k $ is Boltzmann’s constant ($1.38 \times 10^{-23} \, \text{J/K}$),
  - $ T $ is the absolute temperature in Kelvin,
  - $ B $ is the bandwidth of the signal.

- Shot Noise originates in electronic devices due to the discrete nature of electric charge, particularly in semiconductor devices like diodes and transistors. It becomes significant at very low signal levels, such as in photodetectors or when detecting weak signals.

- Intermodulation Noise occurs when multiple signals mix within a nonlinear component, creating new frequencies that interfere with the original signal. This is a common phenomenon in amplifiers and systems handling multiple frequencies, such as in cellular or satellite communication.

- Noise due to crosstalk arises when the signals from one communication channel interfere with another. It is common in wired communication systems such as telephone lines or twisted pair cables.

- Environmental noise includes electromagnetic interference from nearby electrical equipment, power lines, or lightning strikes, and human-made sources like radio transmitters or industrial machinery.



In [ ]:
# --- Demonstration: generate simple noise ---
fs = 1000  # Sampling frequency [Hz]
t = np.linspace(0, 1, fs)  # Time vector [s]
signal = np.sin(2 * np.pi * 5 * t)  # Original signal: 5 Hz sine wave
noise = np.random.normal(0, 0.5, fs)  # Additive white Gaussian noise
noisy_signal = signal + noise

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t, noisy_signal, label="Signal + Noise", alpha=0.7)
ax.plot(t, signal, label="Original Signal", lw=2)
ax.set_xlabel("Time [s]")
ax.set_ylabel("Amplitude")
ax.set_title("Example of Additive White Gaussian Noise")

# Dark background styling
ax.set_facecolor((0.2, 0.2, 0.3))
ax.legend(fancybox=True, framealpha=1, loc="upper right", facecolor=(0.6, 0.61, 0.8))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2. Impact of Noise on Received Data

Noise distorts the transmitted signal, causing errors in the received data. The extent of the distortion depends on the **signal-to-noise ratio (SNR)**, which measures the power of the signal relative to the noise power.

SNR is defined as the ratio of the signal power ($ P_s $) to the noise power ($ P_n $). A higher SNR indicates a clearer signal with less noise interference, resulting in better data reception. A lower SNR means the signal is more corrupted by noise, leading to data errors. SNR is often expressed in decibels which makes it easier to compare large or small values:

$$
\text{SNR} = \frac{P_s}{P_n}
\qquad
\text{SNR}_{\text{dB}} = 10\log_{10}\left(\frac{P_s}{P_n}\right)
$$

This gives some useful intuition:
- 0 dB: signal and noise have equal power
- positive SNR: signal power is larger than noise power
- negative SNR: noise power is larger than signal power
- every increase of 10 dB corresponds to a 10 times larger signal-to-noise power ratio

The **Bit Error Rate (BER)** measures the fraction of bits received incorrectly over a communication link due to noise. A higher BER signifies poorer communication quality. Noise increases BER by distorting the signal’s amplitude, phase, or frequency, making it harder for the receiver to correctly interpret the transmitted bits.



In [ ]:
# --- Example: Compare two noisy received signals with different SNRs ---

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, interact

# Signal power
P_signal = np.mean(signal ** 2)

# Fixed noise shapes so only the strength changes when moving sliders
rng = np.random.default_rng(2025)
base_noise_1 = rng.standard_normal(size=signal.shape)
base_noise_2 = rng.standard_normal(size=signal.shape)


def plot_two_noisy_signals(snr_db_1=3.0, snr_db_2=0.0):
    # Convert SNR from dB to linear
    snr_lin_1 = 10 ** (snr_db_1 / 10)
    snr_lin_2 = 10 ** (snr_db_2 / 10)

    # Choose noise powers to achieve the requested SNRs
    P_noise_1 = P_signal / snr_lin_1
    P_noise_2 = P_signal / snr_lin_2

    noise_1 = np.sqrt(P_noise_1) * base_noise_1
    noise_2 = np.sqrt(P_noise_2) * base_noise_2

    noisy_signal_1 = signal + noise_1
    noisy_signal_2 = signal + noise_2

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(t, noisy_signal_1, label=f"Lower SNR ({snr_db_1:.1f} dB)", alpha=0.8)
    ax.plot(t, noisy_signal_2, label=f"Higher SNR ({snr_db_2:.1f} dB)", alpha=0.8)

    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Amplitude")
    ax.set_title("Impact of SNR on Signal Quality")

    # Dark theme styling
    ax.set_facecolor((0.2, 0.2, 0.3))
    ax.legend(
        fancybox=True,
        framealpha=1,
        loc="upper right",
        facecolor=(0.6, 0.61, 0.8)
    )
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

low_slider = FloatSlider(
    value=0, min=-10, max=5, step=1,
    description="Lower SNR [dB]",
    style={"description_width": "120px"}
)

high_slider = FloatSlider(
    value=10, min=5, max=25, step=1,
    description="Higher SNR [dB]",
    style={"description_width": "120px"}
)

output = interactive_output(
    plot_two_noisy_signals,
    {"snr_db_1": low_slider, "snr_db_2": high_slider}
)

display(VBox([
    HBox([low_slider, high_slider]),
    output
]))

### 3. Channel Capacity $C$

The Shannon-Hartley theorem defines the maximum rate at which information can be transmitted over a communication channel with a given bandwidth and noise level, while still being recoverable without error.

$C = B \log_2\left( 1 + \frac{P_s}{P_n} \right) $

Where:
  - $ C $ is the channel capacity in bits per second (b/s),
  - $ B $ is the bandwidth in Hertz (Hz),
  - $ P_s $ is the signal power,
  - $ P_n $ is the noise power. The noise is assumed to be Gaussian (see also below, AWGN).

The equation shows that the capacity of a communication channel is dependent on both **bandwidth** and **SNR**. Increasing the bandwidth allows more data to be transmitted simultaneously, while a higher SNR increases the channel capacity because the signal becomes easier to distinguish from the noise. Shannon’s theorem indicates that there is a fundamental limit to how much data can be transmitted reliably, even if advanced coding techniques are used. This is useful in designing modems, wireless systems, and wireless communication standards (like Wi-Fi, 5G, etc.).

In practical systems, increasing bandwidth or SNR can be costly. For example, increasing the transmit power to improve SNR may cause interference with other systems, while increasing bandwidth may be constrained by regulatory limits or channel availability.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_capacity(bandwidth_mhz=1.0, snr_db=10.0):
    B = bandwidth_mhz * 1e6
    snr_linear = 10 ** (snr_db / 10)

    C = B * np.log2(1 + snr_linear)

    # Curve over a wider SNR range
    snr_range_db = np.linspace(-10, 30, 200)
    snr_range_linear = 10 ** (snr_range_db / 10)
    C_curve = B * np.log2(1 + snr_range_linear) / 1e6

    fig, ax = plt.subplots(figsize=(6, 3))

    ax.plot(snr_range_db, C_curve, label="Channel capacity")
    ax.scatter(
        snr_db,
        C / 1e6,
        s=60,
        label=f"Current: {C/1e6:.2f} Mbps"
    )

    ax.set_xlabel("SNR [dB]")
    ax.set_ylabel("Capacity [Mbps]")
    ax.set_title("Shannon Capacity")
    ax.set_facecolor((0.2, 0.2, 0.3))
    ax.grid(True, alpha=0.3)
    ax.legend(
        fancybox=True,
        framealpha=1,
        loc="upper left",
        facecolor=(0.6, 0.61, 0.8)
    )

    plt.tight_layout()
    plt.show()


interact(
    plot_capacity,
    bandwidth_mhz=FloatSlider(
        value=1.0,
        min=0.1,
        max=20,
        step=0.1,
        description="B [MHz]",
        continuous_update=True
    ),
    snr_db=FloatSlider(
        value=10,
        min=-10,
        max=30,
        step=1,
        description="SNR [dB]",
        continuous_update=True
    )
)


### 4. AWGN (Additive White Gaussian Noise)

AWGN (Additive White Gaussian Noise) is a widely used mathematical model to represent noise in communication systems. It provides a simplified and idealized framework for analyzing the performance of different communication systems in the presence of noise.

**Additive** means that the noise is added directly to the transmitted signal without any other interaction. **White** means that its power spectral density is constant over all frequencies, analogous to white light, which contains all frequencies of the visible spectrum. **Gaussian** means that its amplitude follows a normal distribution. This is based on the central limit theorem, which states that the sum of many independent and random processes tends to follow a Gaussian (normal) distribution, regardless of the individual distributions.

In mathematical terms, AWGN is modeled as a random noise signal $n(t)$ whose amplitude values follow a Gaussian distribution. This noise is added to the transmitted signal $x(t)$, giving the received signal:
$$ y(t) = x(t) + n(t), $$

Gaussian noise is characterized by its **mean** and **variance**. AWGN is usually modeled with zero mean, while the variance determines its noise power. AWGN is used as a baseline model in communication theory for several important reasons.

#### 4.1 AWGN Benefits

The AWGN model is mathematically simple and allows for easier analysis of communication systems. Since it assumes that the noise is independent of the signal and follows a well-defined distribution (Gaussian), performance metrics like Bit Error Rate (BER), Signal-to-Noise Ratio (SNR), and channel capacity can be derived analytically. In many practical communication systems, thermal and electronic noise is often Gaussian in nature. AWGN provides a reasonably accurate approximation of the noise present in many environments, especially in cases where the communication medium (like air or wire) is not too heavily distorted by other effects (such as fading, interference, or multipath propagation).

The AWGN channel serves as a reference model or ideal benchmark for performance evaluation. It represents the simplest form of a noisy communication channel and allows to analyze how well a system can perform under ideal conditions. For example `Jemula802++` assumes AWGN in its noise and interference model. Once the performance of a system is determined under AWGN, other more complex factors such as fading, interference, and multipath can be added to the analysis to see how the system behaves in more realistic environments.

In [ ]:
# --- Visualization: Gaussian noise distribution ---
noise = np.random.normal(0, 1, 10000)

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(noise, bins=50, density=True, color='skyblue', alpha=0.7)
ax.set_title("Gaussian (Normal) Distribution of AWGN Samples")
ax.set_xlabel("Amplitude")
ax.set_ylabel("Probability Density")

# Dark theme styling
ax.set_facecolor((0.2, 0.2, 0.3))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 4.2 Limitations of AWGN
While AWGN is a widely used and powerful tool for analyzing communication systems, it is not always a completely accurate model of the real world. Some limitations include multipath-fading, interference, and non-white noise.

A received signal can experience fading due to reflections, diffraction, and scattering. AWGN does not account for these multipath effects. In real environments, especially in dense networks like cellular systems, interference from other transmitters can dominate the noise. It may have a frequency- and time-dependent structure and does not necessarily follow the white Gaussian noise model. The noise may have higher power at certain frequencies, which violates the white noise assumption. This sort of noise is known as colored noise.

In [ ]:
# --- Comparison: AWGN vs. colored (non-white) noise ---
def colored_noise(size, alpha=2):
    f = np.fft.rfftfreq(size)
    f[0] = f[1]  # Avoid division by zero
    spectrum = 1 / (f ** (alpha / 2))
    noise = np.fft.irfft(np.random.normal(size=len(f)) * spectrum)
    return noise / np.max(np.abs(noise))

white = np.random.normal(0, 1, fs)
colored = colored_noise(fs)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(white[:500], label="White Noise", alpha=0.9)
ax.plot(colored[:500], label="Colored Noise (1/f²)", alpha=0.8)
ax.set_title("AWGN vs. Colored Noise")
ax.set_xlabel("Sample Index")
ax.set_ylabel("Amplitude")

# Dark theme styling
ax.set_facecolor((0.2, 0.2, 0.3))
ax.grid(True, alpha=0.3)
ax.legend(fancybox=True, framealpha=1, loc="upper right", facecolor=(0.6, 0.61, 0.8))

plt.tight_layout()
plt.show()